# TARDIS — Data Cleaning

This notebook cleans and prepares the **SNCF TGV punctuality dataset** for downstream modelling.

The pipeline runs in order — each step depends on the previous one:

| # | Step | What it does |
|---|------|-------------|
| 2 | Load | Read raw CSV, snapshot the original |
| 2.1 | Drop comment columns | Remove free-text columns unusable in a tabular model |
| 3 | Deduplicate | Remove exact duplicate rows |
| 4 | Map missing values | Audit NaN extent before any action |
| 4.1 | Drop critical NaN | Drop rows missing an identifier that cannot be recovered |
| 4.2–4.5 | Standardise types | Parse dates, convert numerics, cast text, normalise labels |
| 4.6–4.7 | Recover delay NaN | Derive missing delay values from algebraic relationships |
| 4.8 | Report remaining NaN | Document intentionally kept nulls |
| 5 | Cleaning summary | Shape delta: raw vs cleaned |
| 6–10 | Feature engineering | Year, month, season, delay category, cancellation/punctuality rates |
| 11–11.3 | Correct impossible values | Negatives, count overflows, delay hierarchy violations |
| 12 | Export dataset | Write `../cleaned_dataset.csv` |
| 13 | Export audit report | Write `../cleaning_report.csv` |

### 1. Imports

In [ ]:
import pandas as pd

> **Tracking** — initialise the audit report accumulator. Each step appends entries to this list; the full report is written to CSV at step 13.

In [ ]:
report = []

### 2. Data loading

Raw CSV uses `;` as separator. We keep an untouched copy (`original_file`) to compute row/column deltas at the end and to populate the audit report.

In [ ]:
df = pd.read_csv("../data/dataset.csv", sep=";")
original_file = df.copy()
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

> **Tracking** — snapshot the raw dataset: total shape, total null count, and per-column null counts. These become the `initial_state` entries in the audit report.

In [ ]:
_initial_null_total = original_file.isnull().sum().sum()
_perfect_rows       = original_file.dropna().shape[0]
_perfect_cols       = original_file.dropna(axis=1).shape[1]

report += [
    {'metric': 'initial_rows',               'value': original_file.shape[0],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_columns',            'value': original_file.shape[1],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_null_total',         'value': int(_initial_null_total),               'category': 'initial_state', 'reason': 'Total missing values across all cells'},
    {'metric': 'perfect_rows',               'value': _perfect_rows,                          'category': 'initial_state', 'reason': 'Rows with zero NaN — no cleaning needed'},
    {'metric': 'rows_lost_if_strict_dropna', 'value': original_file.shape[0] - _perfect_rows, 'category': 'initial_state', 'reason': 'Rows lost with a strict zero-NaN policy'},
    {'metric': 'perfect_columns',            'value': _perfect_cols,                          'category': 'initial_state', 'reason': 'Columns with zero NaN'},
    {'metric': 'cols_lost_if_strict_dropna', 'value': original_file.shape[1] - _perfect_cols, 'category': 'initial_state', 'reason': 'Columns lost with a strict zero-NaN policy'},
]
for col in original_file.columns:
    report.append({'metric': f'initial_null__{col}', 'value': int(original_file[col].isnull().sum()), 'category': 'initial_null_per_column', 'reason': col})

print(f'Initial null total : {_initial_null_total}')
print(f'Perfect rows       : {_perfect_rows} / {original_file.shape[0]}')
print(f'Perfect columns    : {_perfect_cols} / {original_file.shape[1]}')

### 2.1 Dropping comment columns

Columns whose name ends with `comments` contain free-text justifications written by operators. They cannot be encoded or aggregated in a tabular model, so we drop them entirely.

In [ ]:
comment_cols = df.filter(regex="comments$").columns.tolist()
df = df.drop(columns=comment_cols)
print(f'Dropped {len(comment_cols)} column(s): {comment_cols}')
print(f'Remaining: {df.shape[1]} columns')

> **Tracking** — log each dropped column by name.

In [ ]:
for col in comment_cols:
    report.append({'metric': 'column_dropped', 'value': col, 'category': 'cleaning', 'reason': 'Irrelevant data — free text comment column'})
report.append({'metric': 'columns_dropped_total', 'value': len(comment_cols), 'category': 'cleaning', 'reason': 'Irrelevant data — free text columns removed'})
print(f'Tracked: {len(comment_cols)} column(s) logged.')

### 3. Duplicate check

Duplicate rows artificially inflate all aggregated statistics (means, totals, distributions). We check and remove exact duplicates before any other cleaning step.

In [ ]:
nb_dup = df.duplicated().sum()
print(f'Duplicates found: {nb_dup}')
if nb_dup > 0:
    df = df.drop_duplicates()
    print(f'Removed. Rows remaining: {len(df)}')
else:
    print('No duplicates found.')

> **Tracking** — log the number of duplicate rows removed.

In [ ]:
report.append({'metric': 'rows_dropped_duplicates', 'value': int(nb_dup), 'category': 'cleaning', 'reason': 'Duplicate rows — same observation counted more than once'})
print(f'Tracked: {nb_dup} row(s) logged.')

### 4. Missing values — overview

Before taking any action, we map the full extent of missing data. This guides the strategy for each column: **drop the row** (step 4.1), **derive the value** from existing row data (steps 4.6–4.7), or **leave it null** intentionally (step 4.8).

In [ ]:
nan_counts = df.isnull().sum()
print('Missing values per column:')
print(nan_counts[nan_counts > 0].to_string())
print(f'\nTotal: {nan_counts.sum()}')

### 4.1 Dropping rows with missing critical identifiers

A row is dropped only when one of these five columns is null:
`Date`, `Service`, `Departure station`, `Arrival station`, `Number of scheduled trains`.

Without these, a row cannot be placed in time, attributed to a route, or used to compute any rate — it is unrecoverable. All other NaN values are handled later.

In [ ]:
before = len(df)
df = df.dropna(subset=["Date", "Service", "Departure station", "Arrival station", "Number of scheduled trains"])
print(f'Rows dropped   : {before - len(df)}')
print(f'Rows remaining : {len(df)}')

> **Tracking** — log the number of rows removed due to missing critical identifiers.

In [ ]:
_crit_dropped = before - len(df)
report.append({'metric': 'rows_dropped_critical_nan', 'value': _crit_dropped, 'category': 'cleaning', 'reason': 'Unusable for analysis — missing key identifiers (CRITICAL_COLS)'})
print(f'Tracked: {_crit_dropped} row(s) logged.')

### 4.2 Date parsing

The `Date` column contains mixed string formats across years. `format='mixed'` lets pandas infer the format row by row, which is more robust than a fixed strftime pattern. The column is cast to `datetime64`.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format='mixed')
print(f'dtype : {df["Date"].dtype}')
print(f'Range : {df["Date"].min().date()} → {df["Date"].max().date()}')

### 4.3 Numeric conversion

All columns except `Date`, `Service`, `Departure station`, and `Arrival station` should be numeric. Some cells use a comma as decimal separator — those are normalised to a dot before conversion. The dtype guard prevents calling `.str` on columns that are already numeric, which would silently produce NaN.

In [ ]:
text_cols = ["Date", "Service", "Departure station", "Arrival station"]
numeric_cols = [c for c in df.columns if c not in text_cols]

for col in numeric_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

still_object = [c for c in numeric_cols if df[c].dtype == object]
if still_object:
    print(f'WARNING — still object after conversion: {still_object}')
else:
    print(f'All {len(numeric_cols)} numeric columns successfully converted.')

### 4.4 String dtype cast

By default, pandas reads text columns as `object`, which allows mixed types (strings, `None`, floats). Casting to `string` enforces a uniform dtype and makes null handling explicit (`pd.NA` instead of `None`).

In [ ]:
str_cols = ["Service", "Departure station", "Arrival station"]
df[str_cols] = df[str_cols].astype('string')
print('Dtypes after cast:')
print(df[str_cols].dtypes.to_string())

### 4.5 Label normalisation

Station and service names can appear with inconsistent casing or surrounding spaces (e.g. `"paris montparnasse"` vs `"PARIS MONTPARNASSE "`). Stripping and uppercasing ensures these are treated as the same entity in groupby operations.

In [ ]:
label_cols = ["Departure station", "Arrival station", "Service"]
for col in label_cols:
    df[col] = df[col].str.strip().str.upper()

for col in label_cols:
    print(f'Unique {col}: {df[col].nunique()}')

### 4.6 Recovering departure delay NaN

The two departure delay columns are algebraically linked. When one is missing but the other and the delayed count are present, the missing value can be derived exactly — no imputation needed:

- `dep_late = dep_all × (scheduled / n_delayed)`  ← when `dep_late` is null
- `dep_all  = dep_late × (n_delayed / scheduled)`  ← when `dep_all` is null

In [ ]:
col_dep_late = "Average delay of late trains at departure"
col_dep_all  = "Average delay of all trains at departure"
col_n_dep    = "Number of trains delayed at departure"

mask = df[col_dep_late].isna() & df[col_dep_all].notna() & (df[col_n_dep] > 0)
_dep_late_filled = int(mask.sum())
df.loc[mask, col_dep_late] = df.loc[mask, col_dep_all] * df.loc[mask, "Number of scheduled trains"] / df.loc[mask, col_n_dep]
print(f'dep_late filled from dep_all : {_dep_late_filled} row(s)')

mask = df[col_dep_all].isna() & df[col_dep_late].notna() & df[col_n_dep].notna()
_dep_all_filled = int(mask.sum())
df.loc[mask, col_dep_all] = df.loc[mask, col_dep_late] * df.loc[mask, col_n_dep] / df.loc[mask, "Number of scheduled trains"]
print(f'dep_all  filled from dep_late : {_dep_all_filled} row(s)')

> **Tracking** — log how many cells were recovered for each departure delay column.

In [ ]:
report += [
    {'metric': 'values_filled_dep_late', 'value': _dep_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average departure delay + scheduled/delayed counts'},
    {'metric': 'values_filled_dep_all',  'value': _dep_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train departure delay + delayed/scheduled counts'},
]
print(f'Tracked: dep_late={_dep_late_filled}, dep_all={_dep_all_filled}')

### 4.7 Recovering arrival delay NaN

Same algebraic recovery as step 4.6, applied to the arrival delay columns.

In [ ]:
col_arr_late = "Average delay of late trains at arrival"
col_arr_all  = "Average delay of all trains at arrival"
col_n_arr    = "Number of trains delayed at arrival"

mask = df[col_arr_late].isna() & df[col_arr_all].notna() & (df[col_n_arr] > 0)
_arr_late_filled = int(mask.sum())
df.loc[mask, col_arr_late] = df.loc[mask, col_arr_all] * df.loc[mask, "Number of scheduled trains"] / df.loc[mask, col_n_arr]
print(f'arr_late filled from arr_all : {_arr_late_filled} row(s)')

mask = df[col_arr_all].isna() & df[col_arr_late].notna() & df[col_n_arr].notna()
_arr_all_filled = int(mask.sum())
df.loc[mask, col_arr_all] = df.loc[mask, col_arr_late] * df.loc[mask, col_n_arr] / df.loc[mask, "Number of scheduled trains"]
print(f'arr_all  filled from arr_late : {_arr_all_filled} row(s)')

> **Tracking** — log how many cells were recovered for each arrival delay column.

In [ ]:
report += [
    {'metric': 'values_filled_arr_late', 'value': _arr_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average arrival delay + scheduled/delayed counts'},
    {'metric': 'values_filled_arr_all',  'value': _arr_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train arrival delay + delayed/scheduled counts'},
]
print(f'Tracked: arr_late={_arr_late_filled}, arr_all={_arr_all_filled}')

### 4.8 Remaining NaN — status

Any NaN still present after steps 4.6–4.7 cannot be derived from existing row data. These are left null intentionally — imputing them would introduce artificial signal. They are preserved as-is in the exported dataset.

In [ ]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if not remaining.empty:
    print('Columns with remaining NaN (left as null intentionally):')
    print(remaining.to_string())
else:
    print('No remaining NaN.')

### 5. Cleaning summary

Shape comparison between the raw dataset and the cleaned dataset after all removal steps.

In [ ]:
print(f'Original : {original_file.shape[0]} rows, {original_file.shape[1]} columns')
print(f'Cleaned  : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Removed  : {original_file.shape[0] - df.shape[0]} rows, {original_file.shape[1] - df.shape[1]} columns')

### 6. Year and month extraction

Extracted from the parsed `Date` column as separate integer columns. Used for temporal grouping, trend analysis, and seasonal encoding (step 7).

In [ ]:
df["year"]  = df["Date"].dt.year
df["month"] = df["Date"].dt.month
print(f'Years  : {sorted(df["year"].unique().tolist())}')
print(f'Months : {sorted(df["month"].unique().tolist())}')

### 7. Season encoding

Northern-hemisphere meteorological seasons mapped from the month integer. Season is a relevant feature: winter brings weather disruptions, summer increases passenger load — both affect delay patterns.

In [ ]:
season_map = {
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'autumn',10:'autumn',11:'autumn',
}
df["season"] = df["month"].map(season_map)
print('Season distribution:')
print(df["season"].value_counts().to_string())

### 8. Delay severity category

Average arrival delay is binned into five ordered categories based on operational thresholds used by SNCF:

| Label | Delay range |
|-------|-------------|
| `early` | < 0 min (arrived early) |
| `on_time` | 0–5 min |
| `slight` | 5–15 min |
| `moderate` | 15–30 min |
| `severe` | > 30 min |

In [ ]:
df["delay_category"] = pd.cut(
    df["Average delay of all trains at arrival"],
    bins=[float('-inf'), 0, 5, 15, 30, float('inf')],
    labels=['early', 'on_time', 'slight', 'moderate', 'severe']
)
print('Delay category distribution:')
print(df["delay_category"].value_counts().sort_index().to_string())

### 9. Cancellation rate

Proportion of scheduled trains that were cancelled: `cancelled / scheduled`.
Ranges from `0.0` (no cancellations) to `1.0` (all trains cancelled).

In [ ]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df["Number of scheduled trains"]
print(f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}')

### 10. Punctuality rate

Proportion of trains that arrived on time: `1 − (delayed_at_arrival / scheduled)`.
A value of `1.0` means no train was delayed; `0.0` means every train was delayed.

In [ ]:
df["punctuality_rate"] = 1 - (df["Number of trains delayed at arrival"] / df["Number of scheduled trains"])
print(f'punctuality_rate — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}')

### 11. Negative value correction

Train count columns cannot be negative by definition — a negative count is a data entry error. Negative values are replaced with the **median of valid (≥ 0) values for the same route** (`Departure station` × `Arrival station`). Using a route-level median preserves the local distribution rather than pulling the value towards a global average.

In [ ]:
count_cols = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]
route_cols = ["Departure station", "Arrival station"]

_neg_fixed_total = 0
for col in count_cols:
    mask = df[col] < 0
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(lambda x: x[x >= 0].median())
        df.loc[mask, col] = route_median[mask]
        _neg_fixed_total += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} negative value(s) → route median')
print(f'Total: {_neg_fixed_total} fix(es)')

> **Tracking** — log the total number of negative count values corrected across all columns.

In [ ]:
report.append({'metric': 'values_fixed_negative', 'value': _neg_fixed_total, 'category': 'data_correction', 'reason': 'Impossible negative count values — replaced by route-level median'})
print(f'Tracked: {_neg_fixed_total} fix(es) logged.')

### 11.1 Consistency — counts vs scheduled trains

`Number of cancelled trains`, `Number of trains delayed at departure`, and `Number of trains delayed at arrival` cannot logically exceed the total number of scheduled trains on the same row. Violations are replaced by the route-level median of valid values.

In [ ]:
counts_vs_scheduled = [
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
]

_cons_count_fixed = 0
for col in counts_vs_scheduled:
    mask = df[col].notna() & (df[col] > df["Number of scheduled trains"])
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(
            lambda x, c=col: x[x <= df.loc[x.index, "Number of scheduled trains"]].median()
        )
        df.loc[mask, col] = route_median[mask]
        _cons_count_fixed += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} value(s) exceeded scheduled → route median')
print(f'Total: {_cons_count_fixed} fix(es)')

> **Tracking** — log the number of values corrected for exceeding the scheduled train count.

In [ ]:
report.append({'metric': 'values_fixed_count_vs_scheduled', 'value': _cons_count_fixed, 'category': 'data_correction', 'reason': 'Counts exceeding scheduled trains — logically impossible'})
print(f'Tracked: {_cons_count_fixed} fix(es) logged.')

### 11.2 Consistency — delay hierarchy

By definition, every train delayed more than 30 min was also delayed more than 15 min, so the count hierarchy `>15min ≥ >30min ≥ >60min` must hold. Violations are corrected **bottom-up** (starting from the highest threshold) using the route-level median of valid values.

In [ ]:
delay_hierarchy = [
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]

_hier_fixed = 0
for i in range(len(delay_hierarchy) - 1):
    col_higher = delay_hierarchy[i + 1]
    col_lower  = delay_hierarchy[i]
    mask = df[col_higher].notna() & df[col_lower].notna() & (df[col_higher] > df[col_lower])
    if mask.any():
        route_median = df.groupby(route_cols)[col_higher].transform(
            lambda x, cl=col_lower: x[x <= df.loc[x.index, cl]].median()
        )
        df.loc[mask, col_higher] = route_median[mask]
        _hier_fixed += int(mask.sum())
        print(f'[FIXED] {col_higher} > {col_lower}: {mask.sum()} row(s)')
print(f'Total: {_hier_fixed} fix(es)')

> **Tracking** — log the number of delay hierarchy violations corrected.

In [ ]:
report.append({'metric': 'values_fixed_delay_hierarchy', 'value': _hier_fixed, 'category': 'data_correction', 'reason': 'Delay hierarchy violation — higher threshold exceeded lower threshold'})
print(f'Tracked: {_hier_fixed} fix(es) logged.')

### 11.3 Recomputing derived rates

`cancellation_rate` and `punctuality_rate` were first computed in steps 9–10 using uncorrected counts. They are recomputed here from the now-corrected source columns to ensure consistency.

In [ ]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df["Number of scheduled trains"]
df["punctuality_rate"]  = 1 - (df["Number of trains delayed at arrival"] / df["Number of scheduled trains"])

print(f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}')
print(f'punctuality_rate  — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}')

### 12. Export — cleaned dataset

Index is reset to 0-based after row removals, then the cleaned dataset is written to `../cleaned_dataset.csv`.

In [ ]:
df = df.reset_index(drop=True)
df.to_csv("../cleaned_dataset.csv", index=False)
print(f'Exported: {len(df)} rows, {len(df.columns)} columns → ../cleaned_dataset.csv')
print(f'Columns: {df.columns.tolist()}')

### 13. Export — cleaning audit report

The final dataset shape and null counts are appended to the report accumulator, which is then written to `../cleaning_report.csv`. This file provides a full audit trail of every transformation applied during the pipeline, with one entry per metric.

In [ ]:
_remaining_null = int(df.isnull().sum().sum())
report += [
    {'metric': 'final_rows',         'value': len(df),                              'category': 'final_state', 'reason': 'Rows after full cleaning pipeline'},
    {'metric': 'final_columns',      'value': len(df.columns),                      'category': 'final_state', 'reason': 'Columns after cleaning + feature engineering'},
    {'metric': 'final_null_total',   'value': _remaining_null,                      'category': 'final_state', 'reason': 'Intentionally kept null — non-derivable columns'},
    {'metric': 'total_rows_removed', 'value': original_file.shape[0] - len(df),     'category': 'final_state', 'reason': 'Total rows removed across all cleaning steps'},
]

report_df = pd.DataFrame(report, columns=['metric', 'value', 'category', 'reason'])
report_df.to_csv("../cleaning_report.csv", index=False)

print(f'Report saved: {len(report_df)} entries → ../cleaning_report.csv')
print(report_df['category'].value_counts().to_string())